# Netflix Content Strategy — Exploratory Data Analysis

**Business Problem:** Analyse Netflix's content library to generate insights that help decide which type of shows/movies to produce and how to grow the business in different countries.

**Dataset:** Netflix Movies & TV Shows (~8800 titles) with supporting cast, director, country, and category tables

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn

---

## 1. Problem Statement & Basic Metrics

Netflix wants to understand:
- What type of content performs well in which countries?
- How has content production changed over the years?
- What is the best time to launch a TV show?
- Which actors/directors dominate different content types?
- Should Netflix focus more on Movies or TV Shows going forward?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

In [ ]:
# Load all tables
df        = pd.read_csv('netflix_titles.csv')
cast_df   = pd.read_csv('netflix_cast.csv')
cat_df    = pd.read_csv('netflix_category.csv')
country_df= pd.read_csv('netflix_countries.csv')
dir_df    = pd.read_csv('netflix_directors.csv')

print('Main dataset shape:', df.shape)
print('Cast table shape:', cast_df.shape)
print('Category table shape:', cat_df.shape)
print('Country table shape:', country_df.shape)
print('Director table shape:', dir_df.shape)

In [ ]:
df.head()

In [ ]:
# Basic metrics
print('Total titles:', df.shape[0])
print('Total features:', df.shape[1])
print('Movies:', df[df['type']=='Movie'].shape[0])
print('TV Shows:', df[df['type']=='TV Show'].shape[0])
print('Year range:', df['release_year'].min(), '-', df['release_year'].max())

## 2. Data Types, Categorical Conversion, Missing Values & Statistical Summary

In [ ]:
df.info()

In [ ]:
# Convert categorical columns
cat_cols = ['type', 'rating']
for col in cat_cols:
    df[col] = df[col].astype('category')
print('Converted to category dtype:', cat_cols)
df.dtypes

In [ ]:
# Missing value count
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = round(df.isnull().sum() / df.shape[0] * 100, 2).sort_values(ascending=False)
pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})

In [ ]:
# Statistical summary
df.describe(include='all')

## 3. Non-Graphical Analysis — Value Counts & Unique Attributes

In [ ]:
# Content type distribution
print('=== Content Type ===')
print(df['type'].value_counts())

print('\n=== Rating Distribution ===')
print(df['rating'].value_counts())

print('\n=== Top 10 Release Years ===')
print(df['release_year'].value_counts().head(10))

In [ ]:
# Unique value counts per column
print('Unique values per column:')
for col in df.columns:
    print(f'  {col}: {df[col].nunique()}')

In [ ]:
# Top countries by content count
print('Top 15 countries by content volume:')
print(country_df['country'].value_counts().head(15))

In [ ]:
# Top genres
print('Top 15 genres:')
print(cat_df['listed_in'].value_counts().head(15))

In [ ]:
# Top directors
print('Top 10 directors:')
print(dir_df['director'].value_counts().head(10))

In [ ]:
# Top cast members
print('Top 10 actors by number of titles:')
print(cast_df['cast'].value_counts().head(10))

## 4. Visual Analysis — Univariate & Bivariate

### 4.1 Continuous Variables

In [ ]:
# Distribution of release year — histogram
plt.figure(figsize=(12, 5))
sns.histplot(df['release_year'], bins=40, kde=True, color='steelblue')
plt.title('Distribution of Release Years')
plt.xlabel('Release Year')
plt.ylabel('Count')
plt.show()
# Insight: Content production accelerated sharply post-2015, peaking around 2018-2019

In [ ]:
# Content added per year trend
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

yearly = df.groupby('year_added')['show_id'].count().reset_index()
yearly.columns = ['year', 'count']
yearly = yearly[yearly['year'] >= 2010]

plt.figure(figsize=(12, 5))
sns.lineplot(data=yearly, x='year', y='count', marker='o', color='steelblue')
plt.title('Netflix Content Added Per Year (2010 onwards)')
plt.xlabel('Year')
plt.ylabel('Titles Added')
plt.show()
# Insight: Massive growth from 2015-2019; slight decline in 2020 likely due to COVID production delays

In [ ]:
# Movies duration distribution
movies = df[df['type'] == 'Movie'].copy()
movies['duration_minutes'] = pd.to_numeric(movies['duration_minutes'], errors='coerce')

plt.figure(figsize=(12, 5))
sns.histplot(movies['duration_minutes'].dropna(), bins=40, kde=True, color='coral')
plt.title('Distribution of Movie Duration (minutes)')
plt.xlabel('Duration (minutes)')
plt.ylabel('Count')
plt.show()
# Insight: Most movies fall between 80-120 minutes — standard feature film range

### 4.2 Categorical Variables — Boxplots & Bar Charts

In [ ]:
# Movies vs TV Shows — pie chart
plt.figure(figsize=(6, 6))
counts = df['type'].value_counts()
plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90,
        colors=['steelblue', 'coral'])
plt.title('Movies vs TV Shows on Netflix')
plt.show()
# Insight: ~70% Movies, ~30% TV Shows — Netflix catalogue is movie-heavy

In [ ]:
# Rating distribution
plt.figure(figsize=(12, 5))
rating_counts = df['rating'].value_counts()
sns.barplot(x=rating_counts.values, y=rating_counts.index, palette='Blues_r')
plt.title('Content Rating Distribution')
plt.xlabel('Number of Titles')
plt.ylabel('Rating')
plt.show()
# Insight: TV-MA and TV-14 dominate — Netflix primarily targets mature audiences

In [ ]:
# Top 15 content-producing countries
top_countries = country_df['country'].value_counts().head(15)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='viridis')
plt.title('Top 15 Content-Producing Countries on Netflix')
plt.xlabel('Number of Titles')
plt.ylabel('Country')
plt.show()
# Insight: USA dominates, followed by India, UK, and Japan — strong opportunity in Asian markets

In [ ]:
# Top 15 genres
top_genres = cat_df['listed_in'].value_counts().head(15)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='magma')
plt.title('Top 15 Content Genres on Netflix')
plt.xlabel('Number of Titles')
plt.ylabel('Genre')
plt.show()
# Insight: International Dramas and Documentaries lead — strong demand for global storytelling

In [ ]:
# Best month to launch a TV Show
tv_shows = df[df['type'] == 'TV Show'].copy()
monthly = tv_shows.groupby('month_added')['show_id'].count().reset_index()
monthly.columns = ['month', 'count']
month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}
monthly['month_name'] = monthly['month'].map(month_names)

plt.figure(figsize=(12, 5))
sns.barplot(data=monthly, x='month_name', y='count', palette='coolwarm')
plt.title('Best Month to Launch a TV Show on Netflix')
plt.xlabel('Month')
plt.ylabel('TV Shows Added')
plt.show()
# Insight: July and December see the highest TV Show additions — aligning with summer and holiday seasons

In [ ]:
# Movies vs TV Shows trend over years
type_year = df.groupby(['year_added', 'type'])['show_id'].count().reset_index()
type_year.columns = ['year', 'type', 'count']
type_year = type_year[type_year['year'] >= 2010]

plt.figure(figsize=(12, 5))
sns.lineplot(data=type_year, x='year', y='count', hue='type', marker='o')
plt.title('Movies vs TV Shows Added Per Year')
plt.xlabel('Year')
plt.ylabel('Titles Added')
plt.legend(title='Content Type')
plt.show()
# Insight: TV Show additions grew faster proportionally after 2018 — Netflix shifting toward series content

In [ ]:
# Top 10 actors across all content
top_cast = cast_df['cast'].value_counts().head(10)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_cast.values, y=top_cast.index, palette='pastel')
plt.title('Top 10 Actors by Number of Netflix Titles')
plt.xlabel('Number of Titles')
plt.ylabel('Actor')
plt.show()

In [ ]:
# Top 10 directors
top_dirs = dir_df['director'].value_counts().head(10)
plt.figure(figsize=(12, 5))
sns.barplot(x=top_dirs.values, y=top_dirs.index, palette='Set2')
plt.title('Top 10 Directors by Number of Netflix Titles')
plt.xlabel('Number of Titles')
plt.ylabel('Director')
plt.show()

## 5. Missing Value & Outlier Treatment

In [ ]:
# Missing value summary before treatment
missing_df = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': round(df.isnull().sum() / df.shape[0] * 100, 2)
}).sort_values('Missing %', ascending=False)
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Treatment strategy:
# - drop rows missing date_added, rating, duration (very small %)
# - rating and date_added are critical for analysis
df.dropna(subset=['date_added', 'rating'], inplace=True)
print('Shape after dropping rows with missing date_added/rating:', df.shape)

In [ ]:
# Outlier check — movie duration
plt.figure(figsize=(10, 4))
movies = df[df['type'] == 'Movie'].copy()
movies['duration_minutes'] = pd.to_numeric(movies['duration_minutes'], errors='coerce')
sns.boxplot(x=movies['duration_minutes'].dropna(), color='lightblue')
plt.title('Movie Duration — Outlier Check (Boxplot)')
plt.xlabel('Duration (minutes)')
plt.show()
# Outliers present (very long movies) but retained — they represent valid content like documentaries

In [ ]:
# Outlier stats for movie duration
movies['duration_minutes'].describe()

## 6. Insights from Non-Graphical & Visual Analysis

### 6.1 Range of Attributes
- **Release year** spans 1925 to 2021 — but 75% of content was released after 2013
- **Movie duration** ranges from under 10 mins to 300+ mins; median ~98 minutes
- **Ratings** cover 14 distinct categories; TV-MA and TV-14 account for ~60% of all content

### 6.2 Distribution & Relationships
- Content additions grew exponentially from 2015 to 2019, then slowed — likely due to COVID production halts
- TV Shows are growing faster than Movies in recent years — Netflix is shifting its strategy toward series
- USA produces ~35% of all Netflix content; India is second with ~10% — showing strong localisation push
- Dramas and International Movies dominate genres, reflecting Netflix's global subscriber base

### 6.3 Key Plot Observations
- **Pie chart:** 70/30 Movie vs TV Show split — but TV Show share is growing year on year
- **Line chart (yearly):** Clear inflection point at 2015 — Netflix's original content push begins
- **Bar chart (countries):** USA, India, UK, Japan, South Korea are the top 5 — invest here
- **Month chart:** July and December peak for TV Show launches — align releases with these windows
- **Genre chart:** International Drama is the highest volume genre — highest cross-market appeal

## 7. Business Insights

1. **TV Shows are the future:** TV Show additions are growing faster than Movies post-2018. Subscribers spend more time on serialised content, improving retention metrics.

2. **India is an underserved high-potential market:** India ranks 2nd in content volume but has 1.4B population. Local language drama and reality content could unlock massive subscriber growth.

3. **Mature content dominates:** TV-MA and TV-14 make up ~60% of the library. Netflix's brand is built on adult content — doubling down on premium adult dramas aligns with this.

4. **Korean and Japanese content punches above its weight:** South Korea and Japan produce comparatively fewer titles but generate disproportionate global engagement (Squid Game, Money Heist). More investment here has asymmetric upside.

5. **July and December are optimal launch windows:** These months see the highest TV Show additions — likely because subscribers have more viewing time. High-profile shows should be launched in these windows.

6. **International Drama is the #1 genre:** It appeals across geographies and language barriers. This should be the primary content investment category.

7. **Documentary content is underrated:** Documentaries rank 2nd in volume and require lower production budgets than scripted dramas — high ROI content category.

## 8. Recommendations

1. **Produce more Indian original series** — India is Netflix's largest non-US market by volume. Invest in Hindi, Tamil, and Telugu language drama series to capture regional subscribers.

2. **Increase Korean and Japanese co-productions** — These markets deliver outsized global virality. Budget 15-20% of APAC content spend toward Korean thrillers and Japanese anime.

3. **Launch flagship TV shows in July or December** — Subscriber viewing time is highest in these months. Save premium titles for these launch windows to maximise first-week viewership.

4. **Invest in International Drama as the primary genre** — It is the highest-volume, cross-market genre. Every regional market produces drama — this category scales globally.

5. **Add more TV-MA original series** — Mature content dominates consumption. Scripted adult dramas have the highest rewatch and word-of-mouth value.

6. **Produce more documentaries** — Lower cost, globally appealing, and already the 2nd most popular genre. True crime and nature documentaries travel well across markets.

7. **Reduce dependency on US-only content** — 35% of content is US-produced but Netflix has 222M global subscribers. Diversifying production geographically will serve the majority of subscribers better.